# Word embeddings extraction and exploration

This notebook documents a workflow for training and exploring **Word2Vec embeddings from a historical Greek corpus**. It was developed for research on the Greek Language Question and is intended to make the computational steps readable to researchers who may not be specialists in natural language processing.
In simple terms, a word embedding represents each word as a point in a numerical space. Words that occur in similar contexts tend to be positioned closer together. This makes it possible to explore how terms are associated within a particular historical corpus rather than relying only on dictionary definitions.

### Important interpretive note

The embeddings describe **patterns of word use within the corpus used to train the model**. A close relationship between two words therefore means that they occur in similar textual contexts in this corpus. It does not by itself establish that the words are synonyms, that an author regarded them as equivalent, or that the relationship has the same meaning outside the corpus.

### Reproducibility note

The notebook contains the analytical code used in the research workflow. File names for input data and saved models may need to be adapted to the user's local directory. The code cells have been retained unchanged in this GitHub version; explanatory Markdown has been added for readability.


## 0. Libraries and text-processing setup

The first cells import the Python libraries used throughout the notebook. These include **pandas** and **NumPy** for data handling, **NLTK** for tokenization and stopwords, **Gensim** for Word2Vec and phrase detection, **scikit-learn** for dimensionality reduction, and **Matplotlib** for plotting. NLTK resources are downloaded when required. A custom Greek stopword list is later combined with NLTK's Greek stopwords so that very frequent grammatical words do not dominate the distributional patterns.


In [ ]:
import re
import numpy as np
import pandas as pd
from pprint import pprint

In [ ]:
#to be able to track progress when applying functions
from tqdm.notebook import tqdm
from tqdm.notebook import tqdm_notebook
tqdm_notebook.pandas()

In [ ]:
# εισαγουμε το Genism apo to opoio tha paroume to LDA model
import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
from gensim.models import CoherenceModel
from gensim.models.phrases import Phraser, Phrases
from gensim.models import Word2Vec
from gensim.models.word2vec import Word2Vec

# spacy for lemmatization
import spacy
import os 

# Plotting tools
import matplotlib.pyplot as plt
%matplotlib inline

# Get the interactive Tools for Matplotlib
%matplotlib notebook
import matplotlib.pyplot as plt
plt.style.use('ggplot')

from sklearn.decomposition import PCA

In [ ]:
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
import string
import re
nltk.download('punkt')
nltk.download('stopwords')
remove_stopwords = True
stop_words = set(stopwords.words('english'))


In [ ]:
#pre_model.most_similar("apple")

## 1. Load the dataset

The example filename below is a placeholder: replace `"'input_file'.csv"` with the path to the corpus file you want to analyse.The analysis uses the chapter information, the original text, and a monotonic version of the authorial text.


In [ ]:
#%pylab inline
import pandas as pd
df = pd.read_csv("'input_file'.csv")

In [ ]:
df.columns

In [ ]:
df.head(2)

In [ ]:
# select metadata and columns with authorial text
df1 = df[['Chapters', 'Text', 'text_monotonic']]

In [ ]:
import unicodedata
import gc
import sys
import re

In [ ]:
# Function to strip accents from characters
def strip_accents(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s)
                   if unicodedata.category(c) != 'Mn')

In [ ]:
# Apply the strip_accents function to the 'text_clean2' column
df1['text_NO_accents'] = df1['text_monotonic'].apply(
    lambda x: strip_accents(x.lower().replace(",", " ")) if pd.notnull(x) else x
)

In [ ]:
# Apply the strip_accents function to the 'text_clean2' column
df1['full_text_NO_accents'] = df1['Text'].apply(
    lambda x: strip_accents(x.lower().replace(",", " ")) if pd.notnull(x) else x
)

In [ ]:
nltk.download('stopwords')
from nltk.tokenize import word_tokenize
nltk.download('punkt')

In [ ]:
#insert greek punctuation
punctL = list(set(string.punctuation))

In [ ]:
#check nltk greek stop words
gr_stop_words = nltk.corpus.stopwords.words('greek')

In [ ]:
# insert custom list of stopwords
custom_stopwords = ['ακομα', 'ακομη', 'αλλα', 'αλλες', 'αλλη', 'αλλης', 'αλλο', 'αλλων', 'αν', 'ανα', 'αναμεσα', 'απ', 'απο', 'αρα', 'αραγε', 'αργα',
 'αρκετα', 'ας', 'αυριο', 'αυτα', 'αυτες', 'αυτη', 'αυτην', 'αυτης', 'αυτο', 'αυτοι', 'αυτον', 'αυτος', 'αυτου', 'αυτους', 'αυτων', 'αυτω', 'αφου', 'γι',
 'για', 'γιατι', 'δα', 'δε', 'δει', 'δεινα', 'δεν', 'δηλαδη', 'δι', 'δια', 'διοτι', 'δις', 'διχως', 'εαν', 'ειδεμη', 'ειθε', 'ειπαν', 'ειπε', 'εις',
 'ειτε', 'εκαστα', 'εκαστες', 'εκαστη', 'εκαστην', 'εκαστης', 'εκαστο', 'εκαστοι', 'εκαστον', 'εκαστος', 'εκαστου', 'εκαστους', 'εκαστων', 'εκει', 'εκεινα',
 'εκεινες', 'εκεινη', 'εκεινην', 'εκεινης', 'εκεινο', 'εκεινοι', 'εκεινον', 'εκεινος', 'εκεινου', 'εκεινους', 'εκεινων', 'εκτος', 'εμας', 'εμεις', 'εμενα',
 'εμπρος', 'εν', 'ενα', 'εναν', 'ενας', 'ενος', 'εντελως', 'εντος', 'εντωμεταξυ', 'ενω', 'ενωπιον', 'εξ', 'εξαφνα', 'εξης', 'εξισου', 'εξω', 'επανω',
 'επειδη', 'επειτα', 'επι', 'επιπλεον', 'επισης', 'επομενως', 'εσας', 'εσεις', 'εσενα', 'εστω', 'εσυ', 'ετερα', 'ετεραι', 'ετερας', 'ετερες', 'ετερη',
 'ετερης', 'ετερο','ετεροι', 'ετερον', 'ετερος', 'ετερου', 'ετερους', 'ετερων', 'ετουτα', 'ετουτες', 'ετουτη', 'ετουτην', 'ετουτης', 'ετουτο', 'ετουτοι',
 'ετουτον', 'ετουτος', 'ετουτου', 'ετουτους', 'ετουτων', 'ετσι', 'ευθυς', 'εφεξης', 'εχθες', 'εχτες', 'εχω', 'εως', 'η', 'ηδη', 'ημιν', 'ημων', 'ην','ητοι',
 'ητις' 'ηττον', 'θα', 'ιδια', 'ιδιαν', 'ιδιας', 'ιδιες', 'ιδιο', 'ιδιοι', 'ιδιον', 'ιδιος', 'ιδιου', 'ιδιους', 'ιδιων', 'ιδιως', 'ισαμε', 'ισια',
 'ισως', 'καθε', 'καθεμια', 'καθεμιας', 'καθενα', 'καθενας', 'καθενος', 'καθετι', 'καθολου', 'καθως', 'και', 'καμια', 'καμιαν', 'καμιας', 'καν', 'κανεν',
 'κανενα', 'κανεναν', 'κανενας', 'κανενος', 'κανετε', 'κανουμε', 'κανουν', 'καποια', 'καποιαν', 'καποιας', 'καποιες', 'καποιο', 'καποιοι', 'καποιον',
 'καποιος', 'καποιου', 'καποιους', 'καποιων', 'καποτε', 'καπου', 'καπως', 'κατ', 'κατι', 'κατιτι', 'κατοπιν', 'κατω', 'κι', 'κιολας', 'κλπ', 'κοντα',
 'κτλ', 'κυριως', 'λογω', 'λοιπα', 'λοιπον', 'μα', 'μας', 'μαζι', 'μακαρι', 'μαλιστα', 'μαλλον', 'μας', 'με', 'μεθαυριο', 'μεμιας', 'μεν', 'μερικα',
 'μερικες', 'μερικοι', 'μερικους', 'μερικων', 'μεσα', 'μεσω', 'μετ', 'μετα', 'μεταξυ', 'μεχρι', 'μη', 'μηδε', 'μην', 'μηπως', 'μητε', 'μια', 'μιαν',
 'μιας', 'μολις', 'μολονοτι', 'μονομιας', 'μου', 'να', 'ναι', 'ο', 'οι', 'ολα', 'ολες', 'ολη', 'ολην', 'ολης', 'ολο', 'ολοι', 'ολον', 'ολονεν', 'ολος',
 'ολου', 'ολους', 'ολων', 'ολως', 'ολωσδιολου', 'ομως', 'οποια', 'οποιαδηποτε', 'οποιαν', 'οποιανδηποτε', 'οποιας', 'οποιασδηποτε', 'οποιδηποτε', 'οποιες', 'οποιεσδηποτε',
 'οποιο', 'οποιοδηποτε', 'οποιοι', 'οποιον', 'οποιονδηποτε', 'οποιος', 'οποιοσδηποτε', 'οποιου', 'οποιουδηποτε', 'οποιους', 'οποιουσδηποτε', 'οποιων',
 'οποιωνδηποτε', 'οποτε', 'οποτεδηποτε', 'οπου', 'οπουδηποτε', 'οπως', 'ορισμενα', 'ορισμενες', 'ορισμενων', 'ορισμενως', 'οσα', 'οσαδηποτε',
 'οσες', 'οσεσδηποτε', 'οση', 'οσηδηποτε', 'οσην', 'οσηνδηποτε', 'οσης', 'οσησδηποτε', 'οσο', 'οσοδηποτε', 'οσοι', 'οσοιδηποτε', 'οσον', 'οσονδηποτε',
 'οσος', 'οσοσδηποτε', 'οσου', 'οσουδηποτε', 'οσους', 'οσουσδηποτε', 'οστις', 'οσων', 'οσωνδηποτε', 'οταν', 'οτε',  'οθεν', 'οτι', 'οτιδηποτε',
 'οτου', 'ου', 'ουδε', 'ουτε', 'παντα', 'παντοτε', 'παντου', 'παντως', 'παρ', 'παρα', 'πει', 'περα', 'περι', 'περιπου', 'περσι', 'περυσι', 'πια',
 'πιθανον', 'πιο', 'πισω', 'πλαι', 'πλεον', 'πλην', 'ποια', 'ποιαν', 'ποιας', 'ποιες', 'ποιο', 'ποιοι', 'ποιον', 'ποιος', 'ποιου', 'ποιους', 'ποιων',
 'πολλα', 'πολυ', 'ποσες', 'ποση', 'ποσην', 'ποσης', 'ποσο', 'ποσοι', 'ποσος', 'ποσους', 'ποτε', 'που', 'πουθε', 'πουθενα', 'πριν', 'προ',
 'προκειται', 'πρβλ ', 'πρβλ',  'προς', 'προτου', 'προχθες', 'προχτες', 'πρωτα', 'πρωτη', 'πρωτης', 'πρωτυτερα', 'πως', 'σαν', 'σας', 'σε', 'σελ',
 'σεις', 'σημερα', 'σιγα', 'σου', 'στα', 'στη', 'στην', 'στης', 'στις', 'στο', 'στον', 'στου', 'στους', 'στων', 'συν', 'συχνα', 'συχνας', 'συχνες',
 'συχνη', 'συχνην', 'συχνης', 'συχνο', 'συχνοι', 'συχνον', 'συχνος', 'συχνου', 'συχνου', 'συχνους', 'συχνων', 'συχνως', 'σχεδον', 'σωστα', 'τα', 'ταδε',
 'ταυτα', 'ταυτες', 'ταυτη', 'ταυτην', 'ταυτης', 'ταυτο', 'ταυτον', 'ταυτος', 'ταυτου', 'ταυτοχρονα', 'ταυτων', 'τε', 'τελικα', 'τελικως', 'τες', 'τετοια',
 'τετοιαν', 'τετοιας', 'τετοιες', 'τετοιο', 'τετοιοι', 'τετοιον', 'τετοιος', 'τετοιου', 'τετοιους', 'τετοιων', 'τη', 'την', 'της', 'τι', 'τιποτα',
 'τιποτε', 'τις', 'τινας','το', 'τοι', 'τον', 'τος', 'τοσα', 'τοσες', 'τοση', 'τοσην', 'τοσης', 'τοσο', 'τοσοι', 'τοσον', 'τοσος', 'τοσου', 'τοσους', 'τοσων',
 'τοτε', 'του', 'τουλαχιστο', 'τουλαχιστον', 'τους', 'τουτα', 'τουτες', 'τουτη', 'τουτην', 'τουτης', 'τουτο', 'τουτοι', 'τουτοις', 'τουτον', 'τουτος', 'τουτου', 'τουτους',
 'τουτων', 'τυχον', 'των', 'τωρα', 'υπ','υπερ', 'υπο', 'υποψη', 'υποψιν', 'υστερα', 'χθες','χτες', 'χωρις','ψηλα', 'ω', 'ων','ως', 'ωσαν', 'ωσοτου', 'ωσπου',
 'ωστε', 'ωστοσο', 'ωχ']

In [ ]:
#turn the lists to sets
gr_stop_words = set(nltk.corpus.stopwords.words('greek'))
custom_stopwords = set(custom_stopwords)

# Combine the 2 sets and convert the result back to a list
gr_stop_words2 = gr_stop_words.union(custom_stopwords)

#gr_stop_words = list(gr_stop_words.union(custom_stopwords))

In [ ]:
# Function to remove diacritical marks
def remove_accents(input_str):
    nfkd_form = unicodedata.normalize('NFKD', input_str)
    return "".join([c for c in nfkd_form if not unicodedata.combining(c)])


# Apply accent removal to stopwords
cleaned_stop_words = {remove_accents(word) for word in gr_stop_words2}
#cleaned_stop_words

In [ ]:
# Apply to all the stopwords 
cleaned_stop_words = {word.replace("'", "").replace("’", "").replace("^", "") for word in cleaned_stop_words}

# Display the cleaned set of stop words
#cleaned_stop_words

In [ ]:
# Define a function to remove stopwords
def remove_stopwords(text):
    words = word_tokenize(text)
    cleaned_words = [word for word in words if word.lower() not in cleaned_stop_words]
    return ' '.join(cleaned_words)

In [ ]:
# large dataset.... lets keep a progress bar to calculate when we are going to finish
from tqdm import tqdm
tqdm.pandas()

In [ ]:
df1['no_stop'] = df1['text_NO_accents'].progress_apply(remove_stopwords)

### What preprocessing changes and what it does not

At this point, the working text has been lowercased, stripped of diacritics, and filtered using the combined stopword list. These operations reduce orthographic and grammatical variation before model training. They also mean that the vocabulary shown later reflects the **processed representation of the corpus**, not the original spelling found on the printed page.


## 2. Tokenize the text

Before training word embeddings, the text is divided into sentences and then into individual words (tokens). Punctuation is removed and the text is converted to lowercase. The result is a list of tokenized sentences, which is the format expected by the Word2Vec models used below.


In [ ]:
import re
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize

In [ ]:
nltk.download('punkt')

In [ ]:
def preprocess_text(text):
    sentences = sent_tokenize(text)  # Tokenize into sentences
    processed_sentences = []

    for sentence in sentences:
        sentence = re.sub(r'[^\w\s]', '', sentence).lower()  # Clean text
        words = word_tokenize(sentence)  # Tokenize sentence into words
        if words:  # Avoid empty sentences
            processed_sentences.append(words)  

    return processed_sentences  # List of lists (tokenized sentences)

In [ ]:
# Apply preprocessing to each row and collect processed sentences
all_processed_sentences = []

In [ ]:
#df1['no_stop'].progress_apply(lambda x: all_processed_sentences.extend(preprocess_text(x)))
all_processed_sentences = [sent for text in df1['no_stop'].dropna() for sent in preprocess_text(text)]

In [ ]:
# Print the first  processed sentences to check
print(all_processed_sentences[:2]) 

## 3. Detect frequent two-word expressions

Some meanings are carried by combinations of words rather than by single words. The following cells use Gensim's phrase detection to identify recurring two-word expressions (bigrams) and keep them together as single units when appropriate.


In [ ]:
from gensim.models.phrases import Phrases, Phraser
# make bigrams
phrases = Phrases(all_processed_sentences)


In [ ]:
# The Phraser object is used from now on to transform sentences
# Train bigram model
bigram_phrases = Phrases(all_processed_sentences)  
bigram = Phraser(bigram_phrases)

# Apply bigram transformation
all_processed_sentences = [bigram[sent] for sent in all_processed_sentences]

In [ ]:
# check 
all_processed_sentences[2:3]

In [ ]:
total_tokens = sum(len(sentence) for sentence in all_processed_sentences)

print("Total tokens:", total_tokens)

In [ ]:
bigrams_found = {
    token
    for sentence in all_processed_sentences
    for token in sentence
    if "_" in token
}


#check
print("Number of bigrams:", len(bigrams_found))
print(list(bigrams_found)[:30])

## 4. Inspect frequencies and normalize selected forms

Historical Greek contains grammatical and spelling variation. For a small set of words central to this analysis, selected inflected forms are mapped to a common form before the embeddings are trained. The following cells also provide simple frequency checks so that the researcher can see how often selected forms occur in the corpus.


In [ ]:
normalization_map = {
    'δημοτικη': 'δημοτικη',
    'δημοτικην': 'δημοτικη',
    'δημοτικης': 'δημοτικη',

    'καθαρευουσα': 'καθαρευουσα',
    'καθαρευουσαν': 'καθαρευουσα',
    'καθαρευουσης': 'καθαρευουσα',

    'κοινη': 'κοινη',
    'κοινην': 'κοινη',
    'κοινης': 'κοινη',

    'καθωμιλημενης': 'καθωμιλημενη',
'καθωμιλημενην': 'καθωμιλημενη',
'καθωμιλημενω': 'καθωμιλημενη',
'καθωμιλημενον': 'καθωμιλημενη',
'καθαρευον': 'καθαρευον',
'καθαρευοντος': 'καθαρευον',
}

def normalize_targets(tokens):
    return [normalization_map.get(token, token) for token in tokens]

In [ ]:
all_processed_sentences = [
    normalize_targets(sentence)
    for sentence in all_processed_sentences
]

In [ ]:
for target in ['δημοτικη', 'καθαρευουσα', 'κοινη', 
               'καθωμιλημενη', 'καθαρευον']:
    
    count = sum(
        sentence.count(target)
        for sentence in all_processed_sentences
    )
    
    print(target, count)

In [ ]:
text = ' '.join(df1['no_stop'].dropna().astype(str)).lower()


In [ ]:
from collections import Counter

tokens = re.findall(r'\b\w+\b', text.lower())
counts = Counter(tokens)

def stem_frequencies(stem):
    results = {
        word: count
        for word, count in counts.items()
        if word.startswith(stem)
    }
    
    return dict(
        sorted(results.items(), key=lambda x: x[1], reverse=True)
    )

In [ ]:
def stem_frequency_total(stem):
    forms = stem_frequencies(stem)
    
    print(f"Stem: {stem}-")
    print(f"Total occurrences: {sum(forms.values())}")
    print(f"Distinct forms: {len(forms)}")
    print()
    
    for word, freq in forms.items():
        print(f"{word}: {freq}")

In [ ]:
stems = [
    'δημοτικ',
    'καθαρευουσ',
    'καθαρο',
    'λαλουμ',
    'καθομιλουμ',
    'καθωμιλημεν',
    'καθαρευ',
    'κοινη',
    'αρχαια',
    'νεοελληνικ',
    'απλοελληνικ',
    'ελληνικ',
    'χυδαι',
    'μιξοβαρβαρ',
    'βαρβαρο',
    'ξενισμ',
    'γραφομεν',
    'φραγκορωμεικ',
    'αγοραι',
    'απλοελληνικ',
    'βυζαντιν', 
    'γερμανογραικικ', 
    'γραικικ', 
    'γραικοβαρβαρ', 
    'γραικολατινικ', 
    'δημωδ', 
    'διαλεκτ', 
    'εκκλησιαστικ', 
    'ιδιωμ', 
    'κλασικ',
    'κοινη', 
    'μακαρον', 
    'ρωμαικ',
    'τρακαρ', 
    
]

for stem in stems:
    stem_frequency_total(stem)
    print('-' * 40)

## 5. Cosine similarity

Word embeddings represent words as numerical vectors. **Cosine similarity** is a common way of comparing two such vectors: it measures how similar their directions are rather than how large their values are. Words that occur in similar linguistic contexts tend to receive more similar vectors. A cosine similarity closer to `1` indicates greater similarity in the model's vector space; values closer to `0` indicate weaker similarity. The interpretation remains corpus-dependent: similarity tells us that words are used in related contexts, not that they necessarily have the same dictionary meaning.


#### A simple similarity measure

The function below calculates cosine similarity directly from two vectors. It is included here to make the comparison transparent: the dot product of the two vectors is divided by the product of their lengths.


In [ ]:
#define a cosine similarity function

def cosine(u, v):
    #divide the dot product of uand v to measure how much one vector goes in teh direction of the other
    #the norms (norm) are there to calculate the euclidean norms of the vectors u and v and normalize the reducing their length to 1
    return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))

# we divide the dot product of the vectors by the product of their norms to normalize the similarity masure to be between -1 and1

## 6. Train Word2Vec models

The notebook trains two **Word2Vec** models on the same processed corpus. Word2Vec learns a numerical representation for each sufficiently frequent word from the contexts in which that word occurs.
Both models use the skip-gram architecture and 300-dimensional vectors, but they differ in the size of the context window:

- **Model 1 (`model1`)** uses a small window (`window=2`). It focuses on words occurring in very close contexts and is used here to explore relatively local or substitutable relationships.
- **Model 3 (`model5`)** uses a wider window (`window=5`). It captures a broader context and is used here to explore more topical or domain-related relationships.

These are two views of distributional similarity in this particular corpus; they should not be read as fixed linguistic categories.


In [ ]:
import logging
from gensim.models import Word2Vec

In [ ]:
# Configure logging to show gensim's training progress
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

In [ ]:
# train the model... 
#small window ...We look for similar words which are interchangeable, i.e. one word can substitute another in the similar context.
model1 = Word2Vec(sentences=all_processed_sentences, sg = 1, vector_size=300, window=2, min_count=2, workers=4, epochs=20)

# bigger windor ... word similarity equals topical relation between two words. These words talk about the same topic or are terms of the same domain
model5 = Word2Vec(sentences=all_processed_sentences, sg = 1, vector_size=300, window=5, min_count=2, workers=4, epochs=20)


In [ ]:
model1.build_vocab(all_processed_sentences, update=True)
model5.build_vocab(all_processed_sentences, update=True)

In [ ]:
model1.save("Writer_word2vec.model_interch")
model5.save("Writer_word2vec.model_topical")


### Reload saved models

The trained models can be saved to disk and reloaded later. This avoids retraining them every time the notebook is opened, provided the saved model files are available in the working directory.


In [ ]:
# Load the model from directory
model1 = Word2Vec.load("Writer_word2vec.model_interch")
model5 = Word2Vec.load("Kordatos_word2vecWriter_word2vecmodel_topical")


In [ ]:
# Access the vocabulary
vocabulary = model5.wv.key_to_index

# Print some parts of the vocabulary
for word in list(vocabulary)[:20]:  
    print(word)

In [ ]:
print(model5)

In [ ]:
vocab_size = len(model5.wv.key_to_index)
print(f"Vocabulary size: {vocab_size}")

## 7. Test: Inspect nearest neighbours

`most_similar()` retrieves the words whose vectors are closest to a selected target word. These lists are useful for examining the target word's **distributional neighbourhood**: the terms that tend to occur in comparable contexts in the corpus. The similarity score is model-based and corpus-specific. It is best treated as evidence for contextual association that can then be interpreted against the historical text.


In [ ]:
most_similar_words = model1.wv.most_similar('καθαρευουσα', topn=20)
# Display the top 20 most similar words
for word, similarity in most_similar_words:
    print(f"{word}: {similarity}")

In [ ]:
most_similar_words = model5.wv.most_similar('καθαρευουσα', topn=20)
# Display the top 20 most similar words
for word, similarity in most_similar_words:
    print(f"{word}: {similarity}")

In [ ]:
most_similar_words = model1.wv.most_similar('δημοτικη', topn=20) #καθαροτητα, καθαρον, καθαρωτερα καθαρεύον καθαρα ηλλοιωθη
# Display the top 20 most similar words
for word, similarity in most_similar_words:
    print(f"{word}: {similarity}")

In [ ]:
most_similar_words = model5.wv.most_similar('δημοτικη', topn=20) #καθαροτητα, καθαρον, καθαρωτερα καθαρεύον καθαρα ηλλοιωθη
# Display the top 20 most similar words
for word, similarity in most_similar_words:
    print(f"{word}: {similarity}")

In [ ]:
most_similar_words = model5.wv.most_similar('διγλωσσια', topn=20)
# Display the top 20 most similar words
for word, similarity in most_similar_words:
    print(f"{word}: {similarity}")

## 8. Visualize local semantic neighbourhoods

The next plots provide an exploratory view of the words surrounding a selected term in the embedding space. For each target word, the notebook retrieves its nearest neighbours and uses **t-SNE** to project the high-dimensional vectors into three dimensions so that they can be plotted. The plot is a visual aid, not the original embedding space. Distances and positions in a t-SNE figure should therefore be interpreted cautiously; the nearest-neighbour lists and cosine measures above are the more direct representation of similarity in the trained model.


In [ ]:
from sklearn import manifold
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
import pandas as pd
import numpy as np
from gensim.models import Word2Vec

In [ ]:
%matplotlib inline

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.plot([1, 2, 3], [1, 4, 9])
plt.show()

In [ ]:
word = "δημοτικη"

# Ensure your word is in the vocabulary
if word in model1.wv:
    top_words = [word] + [tup[0] for tup in model1.wv.most_similar(word, topn=40)]
    word_vectors = np.array([model1.wv[word] for word in top_words])  # Adjust for Gensim >= 4.0.0

    # TSNE for dimensionality reduction
    tsne = manifold.TSNE(perplexity=20, n_components=3, init='pca', random_state=42)
    X_tsne = tsne.fit_transform(word_vectors)

    # Create a DataFrame
    dtf_ = pd.DataFrame(X_tsne, index=top_words, columns=["x", "y", "z"])
    dtf_["input"] = 0
    dtf_.loc[word, "input"] = 1  # Mark the input word

    # Plotting
    fig = plt.figure(figsize=(12, 12))
    ax = fig.add_subplot(111, projection='3d')
    colors = ['red' if label == word else 'black' for label in top_words]
    ax.scatter(dtf_['x'], dtf_['y'], dtf_['z'], c=colors, s=50)  # Adjust the size of the dots with 's'

    # Text annotation
    for label, row in dtf_[["x", "y", "z"]].iterrows():
        x, y, z = row
        ax.text(x, y, z, s=label, ha='right', va='bottom', fontsize=9)

    ax.set(xlabel=None, ylabel=None, zlabel=None, xticklabels=[], yticklabels=[], zticklabels=[])

    # Create a dynamic filename
    filename = f"Writer_m1_scatterplot_{word}.png"

    # Save the plot
    plt.savefig(filename, format='png', dpi=300)

    # Show the plot
    plt.show()
else:
    print(f"The word '{word}' is not in the model's vocabulary.")

In [ ]:
word = "καθαρευουσα"
# Ensure your word is in the vocabulary
if word in model5.wv:
    top_words = [word] + [tup[0] for tup in model5.wv.most_similar(word, topn=40)]
    word_vectors = np.array([model5.wv[word] for word in top_words])  # Adjust for Gensim >= 4.0.0

    # TSNE for dimensionality reduction
    tsne = manifold.TSNE(perplexity=20, n_components=3, init='pca', random_state=42)
    X_tsne = tsne.fit_transform(word_vectors)

    # Create a DataFrame
    dtf_ = pd.DataFrame(X_tsne, index=top_words, columns=["x", "y", "z"])
    dtf_["input"] = 0
    dtf_.loc[word, "input"] = 1  # Mark the input word

    # Plotting
    fig = plt.figure(figsize=(14, 14))
    ax = fig.add_subplot(111, projection='3d')
    colors = ['red' if label == word else 'black' for label in top_words]
    ax.scatter(dtf_['x'], dtf_['y'], dtf_['z'], c=colors, s=50)  # Adjust the size of the dots with 's'

    # Text annotation
    for label, row in dtf_[["x", "y", "z"]].iterrows():
        x, y, z = row
        ax.text(x, y, z, s=label, ha='right', va='bottom', fontsize=9)

    ax.set(xlabel=None, ylabel=None, zlabel=None, xticklabels=[], yticklabels=[], zticklabels=[])

    # Create a dynamic filename
    filename = f"Writer_m2_scatterplot_{word}.png"

    # Save the plot
    plt.savefig(filename, format='png', dpi=300)

    # Show the plot
    plt.show()
else:
    print(f"The word '{word}' is not in the model's vocabulary.")

## 9. Compare distances between selected words

This section compares two selected words directly. Their **cosine distance** is calculated from the original Word2Vec vectors. A smaller cosine distance means that the two words occupy more similar positions in the model's embedding space. The surrounding nearest neighbours are also projected with t-SNE for visualization. Because the neighbourhoods of the two target words can overlap, the same word may appear in both neighbour lists and therefore more than once in a comparative t-SNE plot. This does not mean that the Word2Vec vocabulary contains two different copies of that word; it is a consequence of how the two neighbourhood lists are assembled for plotting.


In [ ]:
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from sklearn import manifold
import matplotlib.pyplot as plt
from scipy.spatial import distance

### Model 1: smaller context window

The following comparison uses `model1` (`window=2`). The two target words are highlighted together with their nearest neighbours, while the displayed numerical value is their cosine distance in the original Word2Vec space.


In [ ]:
words = ["δημοτικη", "καθαρευουσα"]
#words = ["απλη", "καθαρευουσα"]


In [ ]:
top_n = 20

# Check if both words are in the vocabulary
if all(word in model1.wv for word in words):
    all_words = []
    colors = []
    
    for word in words:
        # Fetch similar words and add the main word
        similar_words = [tup[0] for tup in model1.wv.most_similar(word, topn=top_n)]
        all_words.extend([word] + similar_words)
        
        # Assign colors: 'red' for the first word, 'blue' for the second
        colors.extend(['red'] if word == words[0] else ['blue'])
        colors.extend(['lightcoral'] * top_n if word == words[0] else ['lightblue'] * top_n)

    # Fetch the vectors for all the unique words
    word_vectors = np.array([model1.wv[word] for word in all_words])

    # Calculate cosine distances for each book
    vector_A1 = model1.wv[words[0]]
    vector_A2 = model1.wv[words[1]]
    distance_A = distance.cosine(vector_A1, vector_A2)

    # Dimensionality reduction with t-SNE
    tsne = manifold.TSNE(perplexity=20, n_components=3, init='pca', random_state=42)
    X_tsne = tsne.fit_transform(word_vectors)

    # Create a DataFrame for plotting
    df = pd.DataFrame(X_tsne, index=all_words, columns=["x", "y", "z"])
    df['color'] = colors

    # Plotting
    fig = plt.figure(figsize=(16, 20))
    ax = fig.add_subplot(111, projection='3d')

    # Plot each point in 3D space
    for label, row in df.iterrows():
        x, y, z = row[:3]
        ax.scatter(x, y, z, color=row['color'], s=50)
        # Adjust text placement
        ax.text(x, y, z + 2, label, color='black', fontsize=9, ha='center', va='bottom', zorder=5)

    # Draw a line between the main words
    main_word_coords = df.loc[words]
    ax.plot(main_word_coords['x'], main_word_coords['y'], main_word_coords['z'], 'k-', lw=2)

    # Display the distance information
    distance_text = f"In Writer model_1, the distance between '{words[0]}' and '{words[1]}' is {distance_A:.4f}."
    plt.figtext(0.5, 0.17, distance_text, ha="center", fontsize=12, bbox={"facecolor":"lightblue", "alpha":0.5, "pad":5})

    ax.set(xlabel=None, ylabel=None, zlabel=None, xticklabels=[], yticklabels=[], zticklabels=[])
    
    # Create a dynamic filename
    filename = f"Writer_m1_comparative__{words}.png"

    # Save the plot
    #plt.savefig(filename, format='png', dpi=300)
    
    plt.show()
else:
    missing = [word for word in words if word not in model1.wv]
    print(f"The words {', '.join(missing)} are not in the model's vocabulary.")

### Model 5: wider context window

The same comparison is repeated with `model5` (`window=5`), allowing the relationship between the selected words to be examined under a broader definition of linguistic context.


In [ ]:
words = ["δημοτικη", "καθαρευουσα"]
#words = ["απλη", "καθαρευουσα"]

In [ ]:
top_n = 20

# Check if both words are in the vocabulary
if all(word in model5.wv for word in words):
    all_words = []
    colors = []
    
    for word in words:
        # Fetch similar words and add the main word
        similar_words = [tup[0] for tup in model5.wv.most_similar(word, topn=top_n)]
        all_words.extend([word] + similar_words)
        
        # Assign colors: 'red' for the first word, 'blue' for the second
        colors.extend(['red'] if word == words[0] else ['blue'])
        colors.extend(['lightcoral'] * top_n if word == words[0] else ['lightblue'] * top_n)

    # Fetch the vectors for all the unique words
    word_vectors = np.array([model5.wv[word] for word in all_words])

    # Calculate cosine distances for each book
    vector_B1 = model5.wv[words[0]]
    vector_B2 = model5.wv[words[1]]
    distance_B = distance.cosine(vector_B1, vector_B2)

    # Dimensionality reduction with t-SNE
    tsne = manifold.TSNE(perplexity=20, n_components=3, init='pca', random_state=42)
    X_tsne = tsne.fit_transform(word_vectors)

    # Create a DataFrame for plotting
    df = pd.DataFrame(X_tsne, index=all_words, columns=["x", "y", "z"])
    df['color'] = colors

    # Plotting
    fig = plt.figure(figsize=(14, 20))
    ax = fig.add_subplot(111, projection='3d')

    # Plot each point in 3D space
    for label, row in df.iterrows():
        x, y, z = row[:3]
        ax.scatter(x, y, z, color=row['color'], s=50)
        # Adjust text placement
        ax.text(x, y, z + 2, label, color='black', fontsize=9, ha='center', va='bottom', zorder=5)

    # Draw a line between the main words
    main_word_coords = df.loc[words]
    ax.plot(main_word_coords['x'], main_word_coords['y'], main_word_coords['z'], 'k-', lw=2)

    # Display the distance information
    distance_text = f"In Writer model_5, the distance between '{words[0]}' and '{words[1]}' is {distance_B:.4f}."
    plt.figtext(0.5, 0.17, distance_text, ha="center", fontsize=12, bbox={"facecolor":"limegreen", "alpha":0.5, "pad":5})

    ax.set(xlabel=None, ylabel=None, zlabel=None, xticklabels=[], yticklabels=[], zticklabels=[])
    
    # Create a dynamic filename
    filename = f"Writer_m3_comparative__{words}.png"

    # Save the plot
    plt.savefig(filename, format='png', dpi=300)
    
    plt.show()
else:
    missing = [word for word in words if word not in model5.wv]
    print(f"The words {', '.join(missing)} are not in the model's vocabulary.")

## 10. A broader two-dimensional view with PCA

The final visualization uses **Principal Component Analysis (PCA)** to project a selected set of word vectors into two dimensions. PCA provides a compact overview of relative positions among the chosen words, but, as with any dimensionality-reduction method, the two-dimensional plot is only an approximation of the original 300-dimensional embedding space.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA


In [ ]:
def display_pca_scatterplot(model1, words=None, sample=0, filename="Writer_m1_general_scatterplot.png"):
    if words is None:
        if sample > 0:
            words = np.random.choice(list(model1.wv.key_to_index.keys()), sample)
        else:
            words = list(model1.wv.key_to_index.keys())

    # Filter words to ensure they exist in the model vocabulary
    words = [word for word in words if word in model1.wv]

    # Accessing the word vectors using the updated Gensim 4.0+ API
    word_vectors = np.array([model1.wv[word] for word in words])
    
    # Perform PCA
    pca = PCA(n_components=2)
    twodim = pca.fit_transform(word_vectors)
    
    # Plotting
    plt.figure(figsize=(12, 16))
    plt.scatter(twodim[:, 0], twodim[:, 1], edgecolors='k', c='r')
    for word, (x, y) in zip(words, twodim):
        plt.text(x, y, word, fontsize=9, ha='center', va='center')  # Adjusted text position

    # Save the plot
    plt.savefig(filename, format='png', dpi=300)
    
    # Show the plot
    plt.show()


# Example usage with specified words
words_list = [
    'αγοραια', 
    'απλη', 
    'αρχαια', 'αρχαιζον', 'βυζαντινον',
    'γραφτη', 'γραφομενη', 'γλωσσα',
    'δημοτικη', 
    'δημωδης', 
    'ιδιωμα',
    'εκκλησιαστικη',
    'διαλεκτος', 'εθνικη', 
    'καθαρευουσα', 
    'καθομιλουμενη', 'καθωμιλημενη', 'καθαρευον',
    'κοινη', 
    'λαλουμενη', 
    'νεοελληνικη', 
    'ρωμαικη', 
    'χυδαια', 
]

display_pca_scatterplot(model1, words=words_list)